# Trinity TAGP Task - kaggle_benchmarks v2026

Trinity Attentional Gateway Probe
Tests selective filtering, sustained attention, attention shifting, needle-in-haystack
**Note:** All 2200 items are short factual questions, no MC conversion needed.

In [ ]:
# === CELL 1: Install & Fix ===
!pip install protobuf==5.29.6 --quiet
!pip install -q kaggle-benchmarks kaggle

In [ ]:
# === CELL 2: Imports & Config ===
import os
os.environ["RENDER_SUBRUNS"] = "False"

import kaggle_benchmarks as kbench
import kaggle
import pandas as pd
from dataclasses import dataclass
import glob

print("✅ Imports successful")

In [ ]:
# === CELL 3: Download Dataset ===
!mkdir -p /kaggle/working/datasets

kaggle.api.dataset_download_files(
    'playra/trinity-cognitive-probes-tagp',
    path='/kaggle/working/datasets/',
    unzip=True
)

In [ ]:
# === CELL 4: Load Data ===
csv_files = glob.glob('/kaggle/working/datasets/**/*.csv', recursive=True)
csv_path = [f for f in csv_files if 'tagp_attention.csv' in f][0]

df = pd.read_csv(csv_path)
eval_df = df.rename(columns={"query": "question", "expected_focus": "answer"})

print(f"📊 Loaded {len(eval_df)} TAGP questions")

In [ ]:
# === CELL 5: Structured Output Schema ===
@dataclass
class CognitiveAnswer:
    answer: str

print("✅ Schema defined")

In [ ]:
# === CELL 6: Inner Task (per-item) ===
@kbench.task(name="TAGP Single", store_task=False)
def tagp_single(llm, question: str, expected_answer: str) -> bool:
    prompt = f""Based on the information provided, give a precise answer.

Question: {question}

Answer:"""
    resp = llm.prompt(prompt, schema=CognitiveAnswer)
    matched = match_answer(resp.answer, expected_answer)
    return matched

# Note: Using flexible matching from notebook template
print("✅ Inner task registered")

In [ ]:
# === CELL 7: Outer Benchmark Task ===
@kbench.task(name="Trinity TAGP", description="Trinity Attentional Gateway Probe")
def tagp_benchmark(llm) -> float:
    with kbench.client.enable_cache():
        runs = tagp_single.evaluate(
            llm=[llm],
            evaluation_data=eval_df,
            n_jobs=2,
            timeout=180,
            max_attempts=1,
            remove_run_files=True,
        )
    df_res = runs.as_dataframe()
    valid = df_res[df_res["result"].notna()]
    acc = float(valid["result"].mean())

    kbench.assertions.assert_true(
        True,
        expectation=f"TAGP accuracy: {acc:.2%} ({len(valid)}/{len(eval_df)})"
    )
    return acc

print("✅ Outer benchmark task registered")

In [ ]:
# === CELL 8: Run & Choose ===
run = tagp_benchmark.run()
print(f"\n🏆 Result: {run.result:.2%}")
%choose tagp_benchmark